# Imputing missing PPV in a mine seismic-event catalogue

This notebook evaluates whether measured event properties can estimate missing peak particle velocity (PPV) values. It is an imputation study: it does not forecast future events or predict rockbursts. Estimates are treated as exploratory unless held-out validation supports their use.


## 1. Set up & Imports

In [ ]:
from pathlib import Path
from typing import Any
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LogNorm
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    BaggingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, KFold, RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

WORKBOOK_PATH = Path("Data_01012000_31032016.xlsx")
SHEET_NAME = "Raw_Data"
OUTPUT_DIR = Path("ppv_results")
N_FOLDS = 5
N_REPEATS = 5
RANDOM_SEED = 42
INTERVAL_LEVEL = 0.90  # empirical, not guaranteed-coverage interval

if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(
        f"Set WORKBOOK_PATH to your workbook. Not found: {WORKBOOK_PATH}"
    )
print(f"Workbook: {WORKBOOK_PATH.resolve()}")


def save_plot(fig: Any, filename: str) -> Path:
    """Save a figure as a high-resolution PNG under OUTPUT_DIR/plots."""
    plot_dir = OUTPUT_DIR / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_path = plot_dir / filename
    fig.savefig(plot_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved plot: {plot_path.resolve()}")
    return plot_path


## 2. Data preparation

In [ ]:
FEATURES = ["Mag","X", "Y", "Z", "Moment", "Energy", "NS", "Stress", "Radius", "Energy_Index",]

COLUMN_MAP = { "Mag": "Mag", "X": "X", "Y": "Y", "Z": "Z", "Moment": "Moment", "Energy": "Energy",
               "NS": "NS", "Stress": "Stress", "Radius": "Radius", "Energy.1": "Energy_Index",}

#### COLUMN_MAP translates the names in the spreadsheet into the names used in the analysis. For example, the spreadsheet column Energy.1 becomes Energy_Index. Pandas often adds .1 when a spreadsheet has duplicate column names, so check that this matches your actual workbook.

In [ ]:
def parse_ppv(value: Any) -> float:
    """Parse an exact PPV value without guessing from ranges or qualifiers."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    text = str(value).strip()
    if not text:
        return np.nan

    number_pattern = r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?"

    # Some workbook cells contain a leading code before an explicitly labelled
    # value (for example, "6 PPV = 91.6 mm/s"). Prefer the labelled value.
    labelled = re.search(
        rf"PPV\s*(?:=|:)\s*({number_pattern})\s*(?:mm\s*/\s*s)?\s*$",
        text,
        flags=re.I,
    )
    if labelled:
        return float(labelled.group(1))

    # Censored values, ranges, and approximate values are not exact targets.
    if re.search(
        r"(?:<=|>=|<|>|≤|≥|~|≈|\bapprox\.?\b|\bless than\b|\bgreater than\b)",
        text,
        flags=re.I,
    ):
        return np.nan

    matches = re.findall(number_pattern, text)
    if len(matches) != 1:
        return np.nan
    return float(matches[0])


The reader loads the worksheet, removes rows without an event number, standardizes selected columns, and parses PPV. The audit cells below report how many raw target values were parsed, missing, or need manual review.


In [ ]:
def read_events(path: Path, sheet_name: str) -> pd.DataFrame:
    """Read the event sheet and create the standardized analysis columns."""
    if not path.is_file():
        raise FileNotFoundError(f"Workbook does not exist: {path}")

    raw = pd.read_excel(path, sheet_name=sheet_name, header=2, engine="openpyxl")
    if raw.shape[1] < 18:
        raise ValueError(
            f"Expected at least 18 columns in {sheet_name!r}; found {raw.shape[1]}. "
            "Check the worksheet and header row."
        )

    # Retain records with a usable event identifier; the worksheet's first
    # non-data/header rows are excluded here.
    event_numbers = pd.to_numeric(raw.iloc[:, 0], errors="coerce")
    raw = raw.loc[event_numbers.notna()].copy()
    if raw.empty:
        raise ValueError("No event records with a numeric event number were found.")

    raw["Event_No"] = pd.to_numeric(raw.iloc[:, 0], errors="coerce")
    raw["Date_raw"] = raw.iloc[:, 1]
    raw["Time_raw"] = raw.iloc[:, 2]
    raw["Working_Place"] = raw.iloc[:, 13].astype("string")
    raw["W_Place"] = raw.iloc[:, 14]
    raw["XY_Error"] = raw.iloc[:, 15]
    raw["Reef_Relation"] = raw.iloc[:, 16]
    raw["PPV_raw"] = raw.iloc[:, 17]

    for source_name, model_name in COLUMN_MAP.items():
        if source_name not in raw.columns:
            raise ValueError(
                f"Expected source column {source_name!r} was not found. "
                "Check COLUMN_MAP against the workbook headers."
            )
        raw[model_name] = pd.to_numeric(raw[source_name], errors="coerce")

    raw["PPV_mm_s"] = raw["PPV_raw"].map(parse_ppv)

    missing_features = [name for name in FEATURES if raw[name].isna().any()]
    if missing_features:
        print(
            "Some predictor values are missing. Median imputation is fitted "
            f"inside each training fold: {missing_features}",
            file=sys.stderr,
        )

    return raw.reset_index(drop=True)


####The columns assigned with iloc use fixed positions in the worksheet. If the workbooks layout changes, those positions may point to the wrong columns, so verify them against the file.

## 3. Candidate models and metrics

In [ ]:
def candidate_models(seed: int) -> dict[str, Any]:
    """Return the unchanged candidate models for log(PPV) prediction."""
    return {
        "Median baseline": DummyRegressor(strategy="median"),
        "Ridge regression": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            Ridge(alpha=10.0),
        ),
        "RBF support vector regression": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            SVR(C=3.0, epsilon=0.10, kernel="rbf"),
        ),
        "Bagged decision trees": make_pipeline(
            SimpleImputer(strategy="median"),
            BaggingRegressor(
                estimator=DecisionTreeRegressor(
                    min_samples_leaf=4,
                    random_state=seed,
                ),
                n_estimators=300,
                max_samples=0.8,
                max_features=0.8,
                random_state=seed,
                n_jobs=-1,
            ),
        ),
        "Random forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=4,
                max_features=0.8,
                random_state=seed,
                n_jobs=-1,
            ),
        ),
        "Gradient boosting": make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=2.0,
                random_state=seed,
            ),
        ),
    }


####Each pipeline handles missing input values with median imputation. Ridge and support vector regression also scale the features, which is important because those models are sensitive to feature magnitudes. Tree-based models generally do not require scaling. The median baseline gives a simple reference: a more complex model should perform better than always predicting the training-set median.

In [ ]:
def score_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Calculate errors on the original PPV scale and log scale."""
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same length.")
    if len(y_true) == 0:
        raise ValueError("Cannot score an empty set of predictions.")
    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError("Scoring values must be finite.")
    if np.any(y_true <= 0) or np.any(y_pred <= 0):
        raise ValueError("PPV values and predictions must be positive.")

    error = y_pred - y_true
    log_error = np.log(y_pred) - np.log(y_true)
    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1.0 - np.sum(error**2) / ss_total if ss_total > 0 else np.nan

    return {
        "MAE_mm_s": float(np.mean(np.abs(error))),
        "RMSE_mm_s": float(np.sqrt(np.mean(error**2))),
        "Median_AE_mm_s": float(np.median(np.abs(error))),
        "R2": float(r2),
        "RMSLE": float(np.sqrt(np.mean(log_error**2))),
    }


1.   MAE is the average absolute prediction error, in mm/s.
2.   RMSE penalizes large errors more heavily than MAE.
3.   Median absolute error describes the typical error and is less affected by outliers.
4.  R² compares the model with predicting the mean.
5.  RMSLE compares errors on a logarithmic scale, which can be useful when PPV values vary greatly.


Spatial clusters are formed from coordinates without using PPV, then held out with GroupKFold. Treat this as a spatial-transfer stress test; it is not a substitute for checking whether the missing events fall outside the observed data range.


## 4. Feature engineering

#### The transformed columns let the models consider relationships that may not be linear: for example, squared magnitude or a combined magnitude stress effect. These are candidates to evaluate, not guaranteed improvements. You can compare model performance with and without them.

In [ ]:
def evaluate_models(
    x: pd.DataFrame,
    y: np.ndarray,
    models: dict[str, Any],
    repeats: int,
    folds: int,
    seed: int,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Repeated CV with fold-wise scores and held-out log predictions."""
    y = np.asarray(y, dtype=float).reshape(-1)
    if len(x) != len(y):
        raise ValueError("The feature rows and target values are not aligned.")
    if not np.isfinite(y).all() or np.any(y <= 0):
        raise ValueError("Observed PPV targets must be finite and positive.")
    if folds < 2 or repeats < 1:
        raise ValueError("Use at least two folds and one repeat.")
    if len(y) < folds:
        raise ValueError(f"Need at least {folds} observed PPV values; found {len(y)}.")

    cv = RepeatedKFold(n_splits=folds, n_repeats=repeats, random_state=seed)
    log_y = np.log(y)
    result_rows = []
    oof_by_model: dict[str, np.ndarray] = {}

    for name, estimator in models.items():
        # Each event receives one held-out prediction in each repeat.
        log_oof = np.full((repeats, len(y)), np.nan, dtype=float)
        split_metrics = []
        for split_number, (train_idx, test_idx) in enumerate(cv.split(x)):
            repeat_idx = split_number // folds
            fitted = clone(estimator)
            fitted.fit(x.iloc[train_idx], log_y[train_idx])
            pred_log = fitted.predict(x.iloc[test_idx])
            if not np.isfinite(pred_log).all():
                raise ValueError(f"Model {name!r} produced non-finite predictions.")
            log_oof[repeat_idx, test_idx] = pred_log
            pred = np.exp(np.clip(pred_log, -20, 20))
            split_metrics.append(score_predictions(y[test_idx], pred))

        oof_by_model[name] = log_oof
        row: dict[str, Any] = {
            "Model": name,
            "CV_folds": folds,
            "CV_repeats": repeats,
        }
        for metric in split_metrics[0]:
            values = [item[metric] for item in split_metrics]
            row[metric] = float(np.mean(values))
            row[f"{metric}_SD_across_splits"] = (
                float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            )
        result_rows.append(row)

    comparison = pd.DataFrame(result_rows).sort_values("MAE_mm_s").reset_index(drop=True)
    return comparison, oof_by_model


def evaluate_spatial_holdout(
    x: pd.DataFrame,
    y: np.ndarray,
    models: dict[str, Any],
    folds: int,
    seed: int,
) -> pd.DataFrame:
    """Stress-test transfer by holding out clusters of event coordinates."""
    y = np.asarray(y, dtype=float).reshape(-1)
    if not {"X", "Y", "Z"}.issubset(x.columns):
        raise ValueError("Spatial validation requires X, Y, and Z coordinates.")
    if len(x) != len(y) or len(y) < folds or folds < 2:
        raise ValueError("Spatial validation needs aligned rows and at least folds rows.")
    coords = x[["X", "Y", "Z"]].to_numpy(dtype=float)
    if np.isinf(coords).any() or np.isnan(coords).all(axis=0).any():
        raise ValueError("Spatial clustering requires at least some finite values per coordinate.")
    # Fill missing coordinate values for clustering only; model pipelines still
    # perform their own fold-specific median imputation on the original inputs.
    coord_medians = np.nanmedian(coords, axis=0)
    missing = np.isnan(coords)
    coords[missing] = np.take(coord_medians, np.where(missing)[1])
    coord_mean = coords.mean(axis=0)
    coord_sd = coords.std(axis=0)
    coord_sd[coord_sd == 0] = 1.0
    coords_scaled = (coords - coord_mean) / coord_sd
    unique_locations = np.unique(coords_scaled, axis=0).shape[0]
    if unique_locations < folds:
        raise ValueError(
            f"Spatial validation needs at least {folds} unique coordinate combinations; "
            f"found {unique_locations}."
        )
    clusters = KMeans(n_clusters=folds, n_init=20, random_state=seed).fit_predict(coords_scaled)
    if np.unique(clusters).size < folds:
        raise ValueError("KMeans produced fewer spatial groups than the requested folds.")

    splitter = GroupKFold(n_splits=folds)
    log_y = np.log(y)
    rows = []
    for name, estimator in models.items():
        fold_metrics = []
        for train_idx, test_idx in splitter.split(x, log_y, groups=clusters):
            fitted = clone(estimator)
            fitted.fit(x.iloc[train_idx], log_y[train_idx])
            pred_log = fitted.predict(x.iloc[test_idx])
            pred = np.exp(np.clip(pred_log, -20, 20))
            fold_metrics.append(score_predictions(y[test_idx], pred))
        row: dict[str, Any] = {"Model": name}
        for metric in fold_metrics[0]:
            values = [item[metric] for item in fold_metrics]
            row[f"Spatial_{metric}"] = float(np.mean(values))
            row[f"Spatial_{metric}_SD"] = (
                float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            )
        rows.append(row)
    return pd.DataFrame(rows)


def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add candidate nonlinear transforms without using the target."""
    required = {"Stress", "Radius", "Mag"}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"Cannot engineer features; columns are missing: {missing}")
    out = frame.copy()
    out["Stress_log1p"] = np.log1p(out["Stress"].clip(lower=0))
    out["Radius_log1p"] = np.log1p(out["Radius"].clip(lower=0))
    out["Magnitude_squared"] = out["Mag"] ** 2
    out["Magnitude_x_logStress"] = out["Mag"] * out["Stress_log1p"]
    return out


#### 5. Reading the workbook

In [ ]:
events = read_events(WORKBOOK_PATH, SHEET_NAME)
observed_mask = events["PPV_mm_s"].notna() & (events["PPV_mm_s"] > 0)
train = events.loc[observed_mask].copy()
missing = events.loc[events["PPV_mm_s"].isna()].copy()

print(f"Event records: {len(events)}")
print(f"Observed positive PPV values: {len(train)}")
print(f"Missing/unparsed PPV values: {len(missing)}")
print("Target-row co-missing fields:")
for col in ["W_Place", "XY_Error", "Reef_Relation"]:
    print(f"  {col}: {missing[col].isna().sum()} of {len(missing)} missing")
print("PPV summary (mm/s):")
print(train["PPV_mm_s"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(2))
print("\nNote: Radius has a small number of missing predictor entries; imputers are fitted inside each training fold.")


## 6. Data audit and missingness

The raw target audit checks whether PPV cells are blank, parsed, or contain qualified/range values that require manual review. The displayed counts are recalculated from the workbook each time the notebook runs; use those current counts in the thesis rather than a count copied from an earlier run.


In [ ]:
# Audit raw PPV values before treating them as missing.
raw_ppv = events["PPV_raw"]
raw_text = raw_ppv.astype("string").str.strip()
nonblank_raw = raw_ppv.notna() & raw_text.ne("")
parsed_target = events["PPV_mm_s"].notna()
nonpositive_target = parsed_target & (events["PPV_mm_s"] <= 0)
unparsed_nonblank = nonblank_raw & ~parsed_target

print(f"Rows in event catalogue: {len(events)}")
print(f"Positive PPV parsed:     {(parsed_target & (events['PPV_mm_s'] > 0)).sum()}")
print(f"PPV blank/missing:       {events['PPV_mm_s'].isna().sum()}")
print(f"Non-positive PPV:        {nonpositive_target.sum()}")
print(f"Nonblank but unparsed:   {unparsed_nonblank.sum()}")
print(f"Duplicate event numbers: {events['Event_No'].duplicated().sum()}")
if unparsed_nonblank.any():
    print("Review these raw PPV entries before deciding whether they are missing:")
    print(events.loc[unparsed_nonblank, ["Event_No", "PPV_raw"]].head(20).to_string(index=False))

co_missing = [c for c in ["W_Place", "XY_Error", "Reef_Relation"] if c in events]
target_missing = events["PPV_mm_s"].isna()
print("\nFields missing on PPV-missing rows:")
for col in co_missing:
    print(f"  {col}: {events.loc[target_missing, col].isna().sum()} / {target_missing.sum()}")


Compare the predictor distributions for observed and missing PPV cases. Large standardized differences or poor range overlap indicate that estimates for some missing rows would be extrapolations. These checks describe the data; they do not establish why PPV is missing.

In [ ]:
# Standardized mean differences and observed-range overlap by predictor.
rows = []
for col in FEATURES:
    obs = train[col].astype(float).dropna()
    miss = missing[col].astype(float).dropna()
    if obs.empty or miss.empty:
        rows.append({"Feature": col, "Observed_mean": obs.mean(), "Missing_mean": miss.mean(),
                     "SMD": np.nan, "Missing_within_observed_range_pct": np.nan,
                     "Missing_n": len(miss)})
        continue
    pooled_sd = np.sqrt((obs.var(ddof=1) + miss.var(ddof=1)) / 2)
    smd = (miss.mean() - obs.mean()) / pooled_sd if pooled_sd > 0 else np.nan
    in_range = miss.between(obs.min(), obs.max()).mean() * 100
    rows.append({"Feature": col, "Observed_mean": obs.mean(), "Missing_mean": miss.mean(),
                 "SMD": smd, "Missing_within_observed_range_pct": in_range,
                 "Missing_n": len(miss)})

missingness_comparison = pd.DataFrame(rows)
print(missingness_comparison.round(3).to_string(index=False))


In [ ]:
# One diagnostic figure: compare magnitude distributions by target availability.
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(train["Mag"].dropna(), bins=20, alpha=0.60, density=True, label="PPV observed")
ax.hist(missing["Mag"].dropna(), bins=20, alpha=0.60, density=True, label="PPV missing")
ax.set_title("Magnitude distribution by PPV availability", loc="left", weight="bold")
ax.set_xlabel("Magnitude")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
save_plot(fig, "magnitude_by_ppv_availability.png")
plt.show()


### Feature leakage review

Before interpreting any model, verify from the mine’s data dictionary or calculation records that every predictor was recorded independently of PPV. In particular, document the formula and timing for `Energy_Index`, `Energy`, `Moment`, `Stress`, and `Radius`. Remove a feature if it uses PPV, a PPV-derived quantity, or information unavailable for the missing rows. Keep `Event_No` and date out of the predictor list; use date only for the chronological validation below.

### Exploratory data analysis


In [ ]:
from matplotlib.colors import LogNorm

sns.set_theme(style="whitegrid", context="notebook")

# Separate observed, missing, and invalid PPV rows.
# PPV must be positive for the log-scale charts and log-target models.
observed_mask = events["PPV_mm_s"].notna() & (events["PPV_mm_s"] > 0)
missing_mask = events["PPV_mm_s"].isna()
invalid_mask = events["PPV_mm_s"].notna() & (events["PPV_mm_s"] <= 0)

observed = events.loc[observed_mask].copy()
missing = events.loc[missing_mask].copy()

print(f"Observed positive PPV: {observed_mask.sum()}")
print(f"Missing PPV:           {missing_mask.sum()}")
print(f"Non-positive PPV:      {invalid_mask.sum()}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

steps = [
    "Load workbook\n& audit data",
    "Explore data\n& engineer features",
    "Compare models\nwith validation",
    "Predict missing PPV\n& assess uncertainty",
    "Export estimates\n& review",
]

xs = np.linspace(0.09, 0.91, len(steps))

for i, (x_pos, label) in enumerate(zip(xs, steps)):
    ax.text(
        x_pos, 0.5, label,
        ha="center", va="center", fontsize=10,
        transform=ax.transAxes,
        bbox=dict(
            boxstyle="round,pad=0.55",
            facecolor="#EAF2F8",
            edgecolor="#35627A",
            linewidth=1.5,
        ),
    )

    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(xs[i + 1] - 0.085, 0.5),
            xytext=(x_pos + 0.085, 0.5),
            xycoords=ax.transAxes,
            arrowprops=dict(arrowstyle="->", color="#35627A", lw=1.5),
        )

ax.set_axis_off()
ax.set_title("PPV imputation workflow", loc="left", fontsize=13, weight="bold")
plt.tight_layout()
save_plot(fig, "ppv_imputation_workflow.png")
plt.show()

In [ ]:
columns_to_check = [c for c in FEATURES + ["PPV_mm_s"] if c in events.columns]
missing_counts = events[columns_to_check].isna().sum().sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
missing_counts.plot(kind="barh", ax=ax, color="#4C78A8")

ax.set_title("Missing values by variable", loc="left", weight="bold")
ax.set_xlabel("Number of missing values")
ax.set_ylabel("")
plt.tight_layout()
save_plot(fig, "missing_values_by_variable.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    np.log10(observed["PPV_mm_s"]),
    bins=28,
    color="#2878B5",
    alpha=0.88,
    edgecolor="white",
)

ax.set_title("Distribution of observed PPV", loc="left", weight="bold")
ax.set_xlabel("log10(PPV in mm/s)")
ax.set_ylabel("Event count")
plt.tight_layout()
save_plot(fig, "observed_ppv_distribution.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(
    observed["Event_No"],
    observed["PPV_mm_s"],
    s=22,
    alpha=0.65,
    label="Observed PPV",
    color="#2878B5",
)

if missing_mask.any():
    lower_edge = observed["PPV_mm_s"].min()
    ax.scatter(
        events.loc[missing_mask, "Event_No"],
        np.full(missing_mask.sum(), lower_edge),
        marker="x",
        s=35,
        label="PPV missing (shown at lower edge)",
        color="#C44E52",
    )

ax.set_yscale("log")
ax.set_title("PPV by event number", loc="left", weight="bold")
ax.set_xlabel("Event number")
ax.set_ylabel("PPV (mm/s, log scale)")
ax.legend()
plt.tight_layout()
save_plot(fig, "ppv_by_event_number.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

positive_ppv = observed["PPV_mm_s"]
if len(positive_ppv) > 0 and positive_ppv.nunique() > 1:
    color_norm = LogNorm(
        vmin=positive_ppv.min(),
        vmax=positive_ppv.max(),
    )
else:
    color_norm = None

points = ax.scatter(
    observed["X"],
    observed["Y"],
    c=observed["PPV_mm_s"],
    s=28,
    cmap="viridis",
    norm=color_norm,
    alpha=0.85,
    label="Observed PPV",
)

if missing_mask.any():
    ax.scatter(
        missing["X"],
        missing["Y"],
        marker="x",
        s=38,
        color="#D62728",
        label="PPV missing",
    )

fig.colorbar(points, ax=ax, label="Observed PPV (mm/s)")
ax.set_title("Event locations and observed PPV", loc="left", weight="bold")
ax.set_xlabel("X coordinate")
ax.set_ylabel("Y coordinate")
ax.legend()
plt.tight_layout()
save_plot(fig, "event_locations_and_observed_ppv.png")
plt.show()

In [ ]:
correlation_data = observed[FEATURES].copy()
correlation_data["log10_PPV"] = np.log10(observed["PPV_mm_s"])

correlations = correlation_data.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlations,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    ax=ax,
    cbar_kws={"shrink": 0.8},
)

ax.set_title("Predictor and log-PPV correlations", loc="left", weight="bold")
ax.tick_params(axis="x", labelrotation=55, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
save_plot(fig, "predictor_log_ppv_correlations.png")
plt.show()

In [ ]:
features_to_plot = ["Mag", "Stress", "Energy", "Moment", "Radius"]

for feature in features_to_plot:
    if feature not in observed.columns:
        print(f"Skipping {feature}: column not found.")
        continue

    plot_data = observed[[feature, "PPV_mm_s"]].dropna()

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        plot_data[feature],
        plot_data["PPV_mm_s"],
        s=22,
        alpha=0.6,
        color="#2878B5",
    )

    ax.set_yscale("log")
    ax.set_title(f"{feature} versus observed PPV", loc="left", weight="bold")
    ax.set_xlabel(feature)
    ax.set_ylabel("PPV (mm/s, log scale)")
    plt.tight_layout()
    safe_feature = re.sub(r"[^A-Za-z0-9_-]+", "_", feature).strip("_").lower()
    save_plot(fig, f"ppv_vs_{safe_feature}.png")
    plt.show()

## 7. Compare models and feature sets

MAE is the pre-specified selection metric here because it reports a typical absolute error in mm/s and is less dominated by a few extreme events than RMSE. The other metrics remain important diagnostics.


In [ ]:
if train.empty or missing.empty:
    raise ValueError("The analysis needs both observed and missing PPV records.")

y_train = train["PPV_mm_s"].to_numpy(dtype=float)
if not np.isfinite(y_train).all() or np.any(y_train <= 0):
    raise ValueError("Training PPV values must all be finite and greater than zero.")

feature_sets = {
    "Raw": train[FEATURES].astype(float),
    "Engineered": engineer_features(train[FEATURES].astype(float)),
}
models = candidate_models(RANDOM_SEED)
parts = []
oof_by_setup = {}

for feature_label, X_values in feature_sets.items():
    cv_scores, oof = evaluate_models(
        X_values,
        y_train,
        models,
        repeats=N_REPEATS,
        folds=N_FOLDS,
        seed=RANDOM_SEED,
    )
    spatial = evaluate_spatial_holdout(
        X_values, y_train, models, folds=N_FOLDS, seed=RANDOM_SEED
    )
    result = cv_scores.merge(spatial, on="Model", how="left")
    result["Feature_Set"] = feature_label
    parts.append(result)
    for model_name, heldout_logs in oof.items():
        oof_by_setup[(feature_label, model_name)] = heldout_logs

comparison = (
    pd.concat(parts, ignore_index=True)
    .sort_values("MAE_mm_s")
    .reset_index(drop=True)
)
best_row = comparison.iloc[0]
best_name = str(best_row["Model"])
best_feature_set = str(best_row["Feature_Set"])

report_columns = [
    "Feature_Set", "Model", "MAE_mm_s", "RMSE_mm_s", "Median_AE_mm_s",
    "R2", "RMSLE", "Spatial_MAE_mm_s", "Spatial_RMSE_mm_s",
]
print(comparison[report_columns].round(3).to_string(index=False))
print(f"\nSelected by repeated-random-CV MAE: {best_name} / {best_feature_set}")

baseline_rows = comparison.loc[
    (comparison["Model"] == "Median baseline")
    & (comparison["Feature_Set"] == "Raw"),
    "MAE_mm_s",
]
if baseline_rows.empty:
    raise ValueError("The raw median baseline is missing from the comparison.")
baseline_mae = float(baseline_rows.iloc[0])
relative_gain = 1 - best_row["MAE_mm_s"] / baseline_mae if baseline_mae > 0 else 0.0
if best_name == "Median baseline" or relative_gain < 0.20 or best_row["R2"] < 0.10:
    print("CAUTION: evidence of predictive skill over the raw median baseline is modest; treat imputations as exploratory.")
print("Review Energy_Index and engineered relationships for leakage or unjustified assumptions.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
model_order = list(models)
for offset, (label, color) in enumerate([("Raw", "#2878B5"), ("Engineered", "#E1812C")]):
    subset = comparison[comparison["Feature_Set"] == label].set_index("Model").reindex(model_order)
    positions = np.arange(len(model_order)) + (offset - 0.5) * 0.38
    axes[0].bar(positions, subset["MAE_mm_s"], width=0.38, label=label, color=color)
    axes[1].bar(positions, subset["Spatial_MAE_mm_s"], width=0.38, label=label, color=color)
for ax, title in zip(axes, ["Repeated random CV", "Spatial-block CV"]):
    ax.set_xticks(np.arange(len(model_order)))
    ax.set_xticklabels(model_order, rotation=45, ha="right", fontsize=8)
    ax.set(title=title, ylabel="MAE (mm/s; lower is better)")
    ax.legend()
fig.suptitle("Does feature engineering improve validation?", weight="bold")
fig.tight_layout()
save_plot(fig, "model_feature_set_validation_comparison.png")
plt.show()


## 8. Stronger validation: nested selection and time holdout

The repeated cross-validation table compares candidates, but choosing the winner from that same table can make its score optimistic. Nested cross-validation selects the feature set and model using only each outer training split, then evaluates that selection on the untouched outer split. Its cross-fitted interval coverage is a diagnostic, not a coverage guarantee under spatial or temporal shift.


In [ ]:
def nested_selection_validation(
    feature_sets: dict[str, pd.DataFrame],
    y: np.ndarray,
    models: dict[str, Any],
    outer_folds: int = 5,
    inner_folds: int = 3,
    calibration_folds: int = 5,
    seed: int = 42,
    interval_level: float = 0.90,
) -> pd.DataFrame:
    """Estimate selection performance on outer folds not used for selection."""
    y = np.asarray(y, dtype=float).reshape(-1)
    if not 0 < interval_level < 1:
        raise ValueError("interval_level must be between 0 and 1.")
    if len(y) < outer_folds or outer_folds < 2:
        raise ValueError("Not enough observed rows for the requested outer folds.")
    if inner_folds < 2 or calibration_folds < 2:
        raise ValueError("Inner and calibration folds must be at least two.")
    if any(len(frame) != len(y) for frame in feature_sets.values()):
        raise ValueError("Every feature set must align with y.")

    outer_cv = KFold(n_splits=outer_folds, shuffle=True, random_state=seed)
    outer_rows = []
    all_true, all_pred, all_low, all_high = [], [], [], []

    for outer_id, (train_idx, test_idx) in enumerate(outer_cv.split(y), start=1):
        inner_candidates = []
        for feature_label, X_all in feature_sets.items():
            X_outer_train = X_all.iloc[train_idx].reset_index(drop=True)
            inner_scores, _ = evaluate_models(
                X_outer_train,
                y[train_idx],
                models,
                repeats=1,
                folds=inner_folds,
                seed=seed + outer_id,
            )
            inner_candidates.extend(
                (row["MAE_mm_s"], feature_label, row["Model"])
                for _, row in inner_scores.iterrows()
            )

        _, chosen_features, chosen_model = min(inner_candidates, key=lambda item: item[0])
        X_outer_train = feature_sets[chosen_features].iloc[train_idx].reset_index(drop=True)
        X_outer_test = feature_sets[chosen_features].iloc[test_idx].reset_index(drop=True)
        folds_for_calibration = min(calibration_folds, len(train_idx))
        if folds_for_calibration < 2:
            raise ValueError("Too few training rows for interval calibration.")

        # Cross-fitted log residuals estimate an empirical interval scale.
        _, calibration_oof = evaluate_models(
            X_outer_train,
            y[train_idx],
            {chosen_model: models[chosen_model]},
            repeats=1,
            folds=folds_for_calibration,
            seed=seed + 100 + outer_id,
        )
        calibration_logs = calibration_oof[chosen_model]
        abs_log_errors = np.abs(np.log(y[train_idx])[None, :] - calibration_logs)
        abs_log_errors = abs_log_errors[np.isfinite(abs_log_errors)]
        if not len(abs_log_errors):
            raise ValueError("No finite residuals were produced for interval calibration.")
        q = float(np.quantile(abs_log_errors, interval_level, method="higher"))

        fitted = clone(models[chosen_model])
        fitted.fit(X_outer_train, np.log(y[train_idx]))
        pred_log = np.clip(fitted.predict(X_outer_test), -20, 20)
        pred = np.exp(pred_log)
        low, high = np.exp(pred_log - q), np.exp(pred_log + q)
        metrics = score_predictions(y[test_idx], pred)
        coverage = np.mean((y[test_idx] >= low) & (y[test_idx] <= high))
        mean_width = np.mean(high - low)

        outer_rows.append({
            "Outer_fold": outer_id,
            "Selected_feature_set": chosen_features,
            "Selected_model": chosen_model,
            **metrics,
            "Empirical_interval_coverage": float(coverage),
            "Mean_interval_width_mm_s": float(mean_width),
        })
        all_true.extend(y[test_idx])
        all_pred.extend(pred)
        all_low.extend(low)
        all_high.extend(high)

    outer_results = pd.DataFrame(outer_rows)
    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)
    all_low = np.asarray(all_low)
    all_high = np.asarray(all_high)
    pooled_coverage = np.mean((all_true >= all_low) & (all_true <= all_high))
    pooled_metrics = score_predictions(all_true, all_pred)
    print("Outer-fold model choices:")
    print(outer_results[["Outer_fold", "Selected_feature_set", "Selected_model"]].to_string(index=False))
    print("\nPooled outer-fold metrics (less selection-biased estimate):")
    print(pd.Series(pooled_metrics).round(3).to_string())
    print(f"\nHeld-out empirical interval coverage: {pooled_coverage:.3f} (nominal level {interval_level:.0%}; diagnostic only)")
    print(f"Mean interval width: {np.mean(all_high - all_low):.2f} mm/s")
    return outer_results


nested_results = nested_selection_validation(
    feature_sets,
    y_train,
    models,
    outer_folds=5,
    inner_folds=3,
    calibration_folds=5,
    seed=RANDOM_SEED,
    interval_level=INTERVAL_LEVEL,
)


A chronological holdout tests whether performance changes across the catalogue timeline. This is a robustness check for imputation across different periods, not a future-event forecasting objective. It selects the model using only the earlier period and scores the later period once.

In [ ]:
# Chronological holdout: select on earlier records, score the later period once.
def parse_event_date(value: Any) -> pd.Timestamp:
    if pd.isna(value):
        return pd.NaT
    return pd.to_datetime(value, errors="coerce", dayfirst=True)


parsed_dates = train["Date_raw"].map(parse_event_date)
valid_positions = np.flatnonzero(parsed_dates.notna().to_numpy())
valid_positions = valid_positions[
    np.argsort(parsed_dates.iloc[valid_positions].to_numpy())
]

if len(valid_positions) < 25:
    print("Too few parseable event dates for a stable chronological holdout.")
else:
    split_at = int(np.floor(0.80 * len(valid_positions)))
    early_idx, late_idx = valid_positions[:split_at], valid_positions[split_at:]
    if len(early_idx) < 5 or len(late_idx) == 0:
        print("The chronological split is too small for the configured five-fold validation.")
    else:
        time_candidates = []
        for feature_label, X_all in feature_sets.items():
            early_X = X_all.iloc[early_idx].reset_index(drop=True)
            scores, _ = evaluate_models(
                early_X,
                y_train[early_idx],
                models,
                repeats=1,
                folds=5,
                seed=RANDOM_SEED,
            )
            best = scores.iloc[0]
            time_candidates.append((best["MAE_mm_s"], feature_label, str(best["Model"])))

        _, time_feature_set, time_model_name = min(time_candidates, key=lambda item: item[0])
        early_X = feature_sets[time_feature_set].iloc[early_idx].reset_index(drop=True)
        late_X = feature_sets[time_feature_set].iloc[late_idx].reset_index(drop=True)
        time_model = clone(models[time_model_name])
        time_model.fit(early_X, np.log(y_train[early_idx]))
        late_pred = np.exp(np.clip(time_model.predict(late_X), -20, 20))
        print(f"Earlier records: {len(early_idx)}; later holdout: {len(late_idx)}")
        print(f"Selected using earlier records only: {time_model_name} / {time_feature_set}")
        print(pd.Series(score_predictions(y_train[late_idx], late_pred)).round(3).to_string())


## 9. Estimate missing PPV and inspect errors

The selected final model is refit on all observed PPV rows. The empirical ranges below summarize cross-validated log errors and are carried forward as exploratory ranges. Use the untouched outer-fold coverage diagnostic above to judge how they behaved on held-out known PPV cases; do not claim guaranteed 90% coverage for the missing cases.


In [ ]:
if train.empty or missing.empty:
    raise ValueError("Need both observed and missing PPV records before final fitting.")
if np.any(~np.isfinite(y_train)) or np.any(y_train <= 0):
    raise ValueError("Training PPV values must be finite and positive.")
if best_feature_set not in feature_sets or best_name not in models:
    raise ValueError("The selected model or feature set is not available.")

best_model = clone(models[best_name])
X_observed = feature_sets[best_feature_set]
best_model.fit(X_observed, np.log(y_train))
print(f"Fitted {best_name} using the {best_feature_set} feature set.")


In [ ]:
Xmiss_raw = missing[FEATURES].astype(float)

if best_feature_set == "Engineered":
    Xmiss = engineer_features(Xmiss_raw)
else:
    Xmiss = Xmiss_raw

# Keep prediction columns in the same order as training columns.
Xmiss = Xmiss[X_observed.columns]

pred_log = np.clip(best_model.predict(Xmiss), -20, 20)
pred_ppv = np.exp(pred_log)

print(f"Generated {len(pred_ppv)} missing-PPV predictions.")

In [ ]:
if not 0 < INTERVAL_LEVEL < 1:
    raise ValueError("INTERVAL_LEVEL must be between 0 and 1.")

oof_best = oof_by_setup[(best_feature_set, best_name)]

if oof_best.shape[1] != len(y_train):
    raise ValueError("Out-of-fold predictions do not align with y_train.")

absolute_log_errors = np.abs(np.log(y_train)[None, :] - oof_best)
absolute_log_errors = absolute_log_errors[np.isfinite(absolute_log_errors)]

if len(absolute_log_errors) == 0:
    raise ValueError("No finite out-of-fold residuals are available.")

q = float(
    np.quantile(
        absolute_log_errors,
        INTERVAL_LEVEL,
        method="higher",
    )
)

interval_label = int(INTERVAL_LEVEL * 100)

lower_ppv = np.exp(pred_log - q)
upper_ppv = np.exp(pred_log + q)

print(f"Calculated empirical {interval_label}% ranges.")

In [ ]:
predictions = pd.DataFrame({
    "Event_No": missing["Event_No"].to_numpy(),
    "Date_raw": missing["Date_raw"].to_numpy(),
    "Magnitude": missing["Mag"].to_numpy(),
    "X": missing["X"].to_numpy(),
    "Y": missing["Y"].to_numpy(),
    "Z": missing["Z"].to_numpy(),
    "Predicted_PPV_mm_s": pred_ppv,
    f"Empirical_{interval_label}pct_Lower_mm_s": lower_ppv,
    f"Empirical_{interval_label}pct_Upper_mm_s": upper_ppv,
    "Selected_Model": best_name,
    "Selected_Feature_Set": best_feature_set,
    "Interval_note": (
        "Empirical repeated out-of-fold range; coverage is not guaranteed"
    ),
})

predictions.head()

In [ ]:
oof_mean_log = np.nanmean(oof_best, axis=0)
oof_pred = np.exp(oof_mean_log)

if len(oof_pred) != len(y_train):
    raise ValueError("Held-out predictions do not align with y_train.")

In [ ]:
# Compare observed PPV with out-of-fold predictions along the event timeline.
# Out-of-fold values are used so each prediction comes from a model that did
# not train on that event's PPV.
parsed_dates = train["Date_raw"].map(parse_event_date)
valid_time_positions = np.flatnonzero(parsed_dates.notna().to_numpy())
if len(valid_time_positions) == 0:
    raise ValueError("No valid event dates are available for the actual-versus-predicted plot.")

valid_time_positions = valid_time_positions[
    np.argsort(parsed_dates.iloc[valid_time_positions].to_numpy())
]
plot_dates = parsed_dates.iloc[valid_time_positions]
actual_by_time = y_train[valid_time_positions]
predicted_by_time = oof_pred[valid_time_positions]

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.scatter(
    plot_dates,
    actual_by_time,
    s=25,
    alpha=0.70,
    color="#2878B5",
    label="Actual PPV",
)
ax.scatter(
    plot_dates,
    predicted_by_time,
    s=25,
    alpha=0.70,
    marker="x",
    color="#E1812C",
    label="Out-of-fold predicted PPV",
)
ax.set_yscale("log")
ax.set_title("Actual and predicted PPV by event date", loc="left", weight="bold")
ax.set_xlabel("Event date")
ax.set_ylabel("PPV (mm/s, log scale)")
ax.grid(True, which="both", axis="y", alpha=0.25)
ax.legend()
fig.autofmt_xdate(rotation=35, ha="right")
fig.tight_layout()

comparison_metrics = score_predictions(actual_by_time, predicted_by_time)
ax.text(
    0.01,
    0.98,
    f"OOF MAE: {comparison_metrics['MAE_mm_s']:.2f} mm/s\n"
    f"OOF RMSE: {comparison_metrics['RMSE_mm_s']:.2f} mm/s\n"
    f"Events with valid dates: {len(valid_time_positions)}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=9,
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "alpha": 0.85},
)
fig.tight_layout()
save_plot(fig, "actual_vs_predicted_ppv_by_event_date.png")
plt.show()


In [ ]:
event_numbers_all = pd.to_numeric(
    train["Event_No"], errors="coerce"
).to_numpy(dtype=float)

valid_positions = np.flatnonzero(np.isfinite(event_numbers_all))
valid_positions = valid_positions[
    np.argsort(event_numbers_all[valid_positions])
]

event_numbers = event_numbers_all[valid_positions]
actual_ppv = y_train[valid_positions]
predicted_ppv = oof_pred[valid_positions]

fig, ax = plt.subplots(figsize=(12, 5.5))

ax.scatter(
    event_numbers,
    actual_ppv,
    s=25,
    alpha=0.7,
    color="#2878B5",
    label="Actual PPV",
)
ax.scatter(
    event_numbers,
    predicted_ppv,
    s=25,
    alpha=0.7,
    marker="x",
    color="#E1812C",
    label="Out-of-fold predicted PPV",
)

ax.set_yscale("log")
ax.set_title("Actual and predicted PPV by event number")
ax.set_xlabel("Event number")
ax.set_ylabel("PPV (mm/s, log scale)")
ax.grid(True, which="both", axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()

save_plot(fig, "actual_vs_predicted_ppv_by_event_number.png")
plt.show()


In [ ]:
log_residuals = np.log(y_train) - oof_mean_log

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    oof_pred,
    log_residuals,
    alpha=0.55,
    s=25,
    color="#55A868",
)
ax.axhline(0, color="#C44E52", linestyle="--")

ax.set_xscale("log")
ax.set_title("Held-out log residuals", loc="left", weight="bold")
ax.set_xlabel("Held-out predicted PPV (mm/s)")
ax.set_ylabel("log(measured PPV) − log(predicted PPV)")

plt.tight_layout()
save_plot(fig, "heldout_log_residuals.png")
plt.show()

In [ ]:
event_order = np.argsort(predictions["Event_No"].to_numpy())
positions = np.arange(len(event_order))

ordered_predictions = pred_ppv[event_order]
ordered_lower = lower_ppv[event_order]
ordered_upper = upper_ppv[event_order]
ordered_event_numbers = predictions["Event_No"].to_numpy()[event_order]

fig, ax = plt.subplots(figsize=(12, 5))

ax.errorbar(
    positions,
    ordered_predictions,
    yerr=[
        ordered_predictions - ordered_lower,
        ordered_upper - ordered_predictions,
    ],
    fmt="o",
    markersize=4,
    capsize=2,
    color="#2878B5",
    ecolor="#9ABAD0",
)

ax.set_yscale("log")

tick_step = max(1, len(event_order) // 15)
tick_positions = positions[::tick_step]
ax.set_xticks(tick_positions)
ax.set_xticklabels(
    ordered_event_numbers[::tick_step].astype(int),
    rotation=45,
    ha="right",
)

ax.set_title(
    f"Missing PPV estimates and empirical {interval_label}% ranges",
    loc="left",
    weight="bold",
)
ax.set_xlabel("Event number")
ax.set_ylabel("PPV (mm/s, log scale)")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
save_plot(fig, "missing_ppv_estimates_with_ranges.png")
plt.show()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prediction_path = OUTPUT_DIR / "ppv_missing_predictions.csv"
metrics_path = OUTPUT_DIR / "ppv_model_comparison.csv"

predictions.to_csv(prediction_path, index=False)
comparison.to_csv(metrics_path, index=False)

print(f"Saved predictions to: {prediction_path.resolve()}")
print(f"Saved model comparison to: {metrics_path.resolve()}")